## 01 — EDA Univariata

> **Analisi esplorativa univariata del dataset pulito NBA.**  
> Obiettivo: comprendere la distribuzione di ogni variabile in isolamento, identificare pattern anomali,  
> stimare il bilanciamento del target e produrre una baseline di vittorie home.
>
> **Input:** `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv`  
> **Output:** statistiche descrittive, grafici di distribuzione, report outlier.

---

**Struttura del notebook**
1. Setup & caricamento dati
2. Overview statistiche numeriche
3. Distribuzioni variabili di performance
4. Distribuzioni variabili temporali
5. Variabili categoriche
6. Analisi del target (W/L)
7. Ispezione outlier

### 1. Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent.parent
sys.path.append(str(ROOT))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from scipy import stats

import itables.options as opt
from itables import init_notebook_mode, show

from src.loader import load_config

warnings.filterwarnings("ignore")

config = load_config()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

init_notebook_mode(all_interactive=True)
opt.warn_on_undocumented_option = False
opt.maxBytes   = 0
opt.pageLength = 15
opt.lengthMenu = [15, 25, 50, 100]
opt.scrollX    = True
opt.classes    = "display nowrap compact"
opt.style      = "width:100%; font-size:13px;"

# ── Plotting configuration ──────────────────────────────────────────────────
NBA_PALETTE   = ["#1d428a", "#c8102e", "#fdb927", "#007a33", "#552583", "#ce1141"]
ACCENT        = "#1d428a"
NEUTRAL       = "#6c757d"
FIG_W_SINGLE  = 10
FIG_H_SINGLE  = 4
FIG_W_GRID    = 16
FIG_H_ROW     = 4
DPI           = 130

sns.set_theme(style="whitegrid", palette=NBA_PALETTE)
plt.rcParams.update({
    "figure.dpi"      : DPI,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "font.size"       : 11,
    "axes.titlesize"  : 13,
    "axes.labelsize"  : 11,
})

# ── Thresholds ──────────────────────────────────────────────────────────────
SKEW_THRESHOLD = 1.0
KURT_THRESHOLD = 3.0   # excess kurtosis
Z_THRESHOLD    = 3.0

NOTEBOOK_VERSION = "1.0.0"
log.info(f"EDA Univariata v{NOTEBOOK_VERSION} — initialized")

#### 1.1 Caricamento dataset pulito

In [ ]:
start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"
DATASET_VERSION = f"{start_season}_{end_season}"

PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]
CLEAN_FILE    = PROCESSED_DIR / f"cleaned_nba_data_{DATASET_VERSION}.csv"

df = pl.read_csv(
    CLEAN_FILE,
    try_parse_dates=True,
    schema_overrides={"game_id": pl.Int64, "team_id": pl.Int64},
)

log.info(f"Loaded  : {CLEAN_FILE.name}")
log.info(f"Shape   : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
log.info(f"Seasons : {df['season'].n_unique()}  ({df['season'].min()} – {df['season'].max()})")
log.info(f"Teams   : {df['team_abbreviation'].n_unique()}")

#### 1.2 Definizione liste variabili

In [ ]:
# Variabili numeriche — raggruppate per dominio
PERF_VARS = ["pts", "off_rating", "def_rating", "net_rating", "pace", "ts_pct"]

SHOOTING_VARS = ["fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]

ALL_NUM_VARS = ["pts", "plus_minus", "off_rating", "def_rating", "net_rating",
                "pace", "ts_pct", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]

# Variabili categoriche
CAT_VARS = ["wl", "home_away", "season"]

# Soglie anomale per PTS (domain knowledge NBA)
PTS_LOW  = 70
PTS_HIGH = 160

log.info(f"Variabili numeriche  : {len(ALL_NUM_VARS)} — {ALL_NUM_VARS}")
log.info(f"Variabili categoriche: {len(CAT_VARS)} — {CAT_VARS}")

# Preview schema
show(
    pl.DataFrame({
        "colonna"  : df.columns,
        "dtype"    : [str(t) for t in df.dtypes],
        "null_cnt" : [df[c].null_count() for c in df.columns],
        "dominio"  : [
            "id" if c in ("game_id", "team_id") else
            "data" if c == "game_date" else
            "temporale" if c == "season" else
            "categorica" if c in CAT_VARS else
            "numerica" if c in ALL_NUM_VARS else
            "flag"
            for c in df.columns
        ],
    }).to_pandas(),
    caption="Schema del dataset pulito",
)

---
### 2. Overview Variabili Numeriche

In [ ]:
# Compute extended descriptive statistics using scipy for skewness and kurtosis
rows = []
for col in ALL_NUM_VARS:
    series = df[col].drop_nulls().to_numpy().astype(float)
    rows.append({
        "variable" : col,
        "count"    : int(len(series)),
        "mean"     : round(float(np.mean(series)), 4),
        "median"   : round(float(np.median(series)), 4),
        "std"      : round(float(np.std(series, ddof=1)), 4),
        "min"      : round(float(np.min(series)), 4),
        "max"      : round(float(np.max(series)), 4),
        "q25"      : round(float(np.percentile(series, 25)), 4),
        "q75"      : round(float(np.percentile(series, 75)), 4),
        "skewness" : round(float(stats.skew(series)), 4),
        "kurtosis" : round(float(stats.kurtosis(series)), 4),  # excess kurtosis
    })

desc_df = pl.DataFrame(rows)

show(
    desc_df.to_pandas(),
    caption="Statistiche descrittive estese — variabili numeriche",
)

#### 2.1 Variabili con skewness elevata (candidati alla trasformazione log)

In [ ]:
high_skew = desc_df.filter(pl.col("skewness").abs() > SKEW_THRESHOLD)

print(f"Variabili con |skewness| > {SKEW_THRESHOLD}: {high_skew.shape[0]}")
if high_skew.shape[0] > 0:
    show(
        high_skew.select(["variable", "skewness", "kurtosis", "min", "max", "mean", "median"])
              .sort("skewness", descending=True)
              .to_pandas(),
        caption=f"Candidati trasformazione log (|skewness| > {SKEW_THRESHOLD})",
    )
else:
    print("Nessuna variabile presenta skewness critica.")

# Bar chart skewness
skew_vals = desc_df.sort("skewness", descending=True)
colors = [
    "#c8102e" if abs(v) > SKEW_THRESHOLD else ACCENT
    for v in skew_vals["skewness"].to_list()
]

fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, FIG_H_SINGLE))
bars = ax.barh(skew_vals["variable"].to_list(), skew_vals["skewness"].to_list(),
               color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(SKEW_THRESHOLD,  color="#c8102e", linestyle="--", linewidth=1.2, label=f"+{SKEW_THRESHOLD}")
ax.axvline(-SKEW_THRESHOLD, color="#c8102e", linestyle="--", linewidth=1.2, label=f"-{SKEW_THRESHOLD}")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Skewness (Fisher)")
ax.set_title("Skewness per variabile numerica — soglia ±1.0")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

#### 2.2 Variabili con kurtosis elevata (code pesanti)

In [ ]:
high_kurt = desc_df.filter(pl.col("kurtosis") > KURT_THRESHOLD)

print(f"Variabili con kurtosis (excess) > {KURT_THRESHOLD}: {high_kurt.shape[0]}")
if high_kurt.shape[0] > 0:
    show(
        high_kurt.select(["variable", "kurtosis", "skewness", "std"])
              .sort("kurtosis", descending=True)
              .to_pandas(),
        caption=f"Code pesanti — kurtosis excess > {KURT_THRESHOLD}",
    )
else:
    print("Nessuna variabile presenta kurtosis critica.")

# Bar chart kurtosis
kurt_vals = desc_df.sort("kurtosis", descending=True)
colors_k = [
    "#c8102e" if v > KURT_THRESHOLD else ACCENT
    for v in kurt_vals["kurtosis"].to_list()
]

fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, FIG_H_SINGLE))
ax.barh(kurt_vals["variable"].to_list(), kurt_vals["kurtosis"].to_list(),
        color=colors_k, edgecolor="white", linewidth=0.5)
ax.axvline(KURT_THRESHOLD, color="#c8102e", linestyle="--", linewidth=1.2,
           label=f"soglia {KURT_THRESHOLD}")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Kurtosis excess (Fisher)")
ax.set_title("Kurtosis per variabile numerica")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
### 3. Distribuzioni Variabili di Performance

#### 3.1 Istogramma + KDE

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(PERF_VARS) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(FIG_W_GRID, FIG_H_ROW * n_rows))
axes = axes.flatten()

for i, col in enumerate(PERF_VARS):
    ax   = axes[i]
    data = df[col].drop_nulls().to_numpy().astype(float)
    row_info = desc_df.filter(pl.col("variable") == col).row(0, named=True)

    sns.histplot(data, bins=50, kde=True, color=ACCENT,
                 edgecolor="white", linewidth=0.3, ax=ax)

    ax.axvline(row_info["mean"],   color="#c8102e", linewidth=1.5,
               linestyle="--", label=f"mean={row_info['mean']:.1f}")
    ax.axvline(row_info["median"], color="#fdb927", linewidth=1.5,
               linestyle=":",  label=f"med={row_info['median']:.1f}")

    # Annotate PTS anomaly thresholds
    if col == "pts":
        ax.axvline(PTS_LOW,  color="#552583", linewidth=1.2, linestyle="-.",
                   label=f"PTS<{PTS_LOW}")
        ax.axvline(PTS_HIGH, color="#552583", linewidth=1.2, linestyle="-.",
                   label=f"PTS>{PTS_HIGH}")

    ax.set_title(f"{col.upper()}  |  skew={row_info['skewness']:.2f}  kurt={row_info['kurtosis']:.2f}")
    ax.set_xlabel(col.upper())
    ax.set_ylabel("Frequenza")
    ax.legend(fontsize=8, loc="upper right")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuzioni variabili di performance — Istogramma + KDE",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

#### 3.2 Boxplot variabili di performance

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(FIG_W_GRID, FIG_H_ROW * n_rows))
axes = axes.flatten()

for i, col in enumerate(PERF_VARS):
    ax   = axes[i]
    data = df[col].drop_nulls().to_numpy().astype(float)

    bp = ax.boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=ACCENT, alpha=0.7),
                    medianprops=dict(color="#fdb927", linewidth=2),
                    flierprops=dict(marker=".", color="#c8102e",
                                   markersize=3, alpha=0.4),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2))

    if col == "pts":
        ax.axhline(PTS_LOW,  color="#552583", linewidth=1.2, linestyle="-.",
                   label=f"<{PTS_LOW}")
        ax.axhline(PTS_HIGH, color="#552583", linewidth=1.2, linestyle="-.",
                   label=f">{PTS_HIGH}")
        ax.legend(fontsize=8)

    ax.set_title(col.upper())
    ax.set_ylabel(col.upper())
    ax.set_xticks([])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Boxplot variabili di performance", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Count anomali PTS
pts_low_cnt  = df.filter(pl.col("pts") < PTS_LOW).shape[0]
pts_high_cnt = df.filter(pl.col("pts") > PTS_HIGH).shape[0]
print(f"Partite con PTS < {PTS_LOW}  : {pts_low_cnt}")
print(f"Partite con PTS > {PTS_HIGH} : {pts_high_cnt}")

---
### 4. Distribuzioni Variabili Temporali

#### 4.1 Partite per stagione

In [ ]:
games_per_season = (
    df.group_by("season")
      .agg(pl.len().alias("n_game_rows"))
      .sort("season")
)

# Each season has 2 rows per game (home + away) → divide by 2 for unique games
games_per_season = games_per_season.with_columns(
    (pl.col("n_game_rows") // 2).alias("n_games")
)

seasons = games_per_season["season"].to_list()
counts  = games_per_season["n_games"].to_list()
mean_games = np.mean(counts)

fig, ax = plt.subplots(figsize=(FIG_W_GRID, FIG_H_SINGLE + 1))
bar_colors = [
    "#c8102e" if abs(c - mean_games) > 0.15 * mean_games else ACCENT
    for c in counts
]
ax.bar(seasons, counts, color=bar_colors, edgecolor="white", linewidth=0.4)
ax.axhline(mean_games, color="#fdb927", linewidth=1.5, linestyle="--",
           label=f"media={mean_games:.0f}")
ax.set_xlabel("Stagione")
ax.set_ylabel("N. partite")
ax.set_title("Distribuzione partite per stagione (partite uniche)")
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Stagioni outlier (>±15% dalla media {mean_games:.0f}):")
for s, c in zip(seasons, counts):
    if abs(c - mean_games) > 0.15 * mean_games:
        print(f"  {s}: {c} partite  ({(c - mean_games) / mean_games:+.1%} dalla media)")

#### 4.2 Distribuzione partite per mese

In [ ]:
MONTH_LABELS = {
    1: "Gen", 2: "Feb", 3: "Mar", 4: "Apr", 5: "Mag", 6: "Giu",
    7: "Lug", 8: "Ago", 9: "Set", 10: "Ott", 11: "Nov", 12: "Dic"
}

df_month = df.with_columns(
    pl.col("game_date").dt.month().alias("month")
)

games_per_month = (
    df_month.group_by("month")
            .agg((pl.len() // 2).alias("n_games"))
            .sort("month")
)

months = games_per_month["month"].to_list()
month_labels = [MONTH_LABELS[m] for m in months]
month_counts = games_per_month["n_games"].to_list()

fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, FIG_H_SINGLE))
ax.bar(month_labels, month_counts, color=ACCENT, edgecolor="white", linewidth=0.4)
ax.set_xlabel("Mese")
ax.set_ylabel("N. partite")
ax.set_title("Stagionalità intra-stagionale: partite per mese")
plt.tight_layout()
plt.show()

print("Note: la stagione NBA va da ottobre/novembre a giugno (playoffs).")

#### 4.3 Partite per team per stagione — verifica copertura

In [ ]:
games_team_season = (
    df.group_by(["season", "team_abbreviation"])
      .agg(pl.len().alias("n_rows"))
      .sort(["season", "team_abbreviation"])
)

# Pivot: rows=teams, cols=seasons
pivot = (
    games_team_season
    .pivot(on="season", index="team_abbreviation", values="n_rows", aggregate_function="sum")
    .sort("team_abbreviation")
)

pivot_pd = pivot.to_pandas().set_index("team_abbreviation")
pivot_pd = pivot_pd.reindex(sorted(pivot_pd.columns), axis=1)

fig, ax = plt.subplots(figsize=(FIG_W_GRID + 6, max(8, len(pivot_pd) * 0.35)))
sns.heatmap(
    pivot_pd,
    ax=ax,
    cmap="Blues",
    linewidths=0.3,
    linecolor="#e0e0e0",
    annot=False,
    cbar_kws={"label": "N. righe (home+away)"},
    mask=pivot_pd.isna(),
)
ax.set_title("Righe per team per stagione (copertura del dataset)", fontsize=13)
ax.set_xlabel("Stagione")
ax.set_ylabel("Team")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Flag teams with anomalous coverage
season_min = games_team_season.group_by("season").agg(pl.col("n_rows").median().alias("median_rows"))
anomalous = (
    games_team_season
    .join(season_min, on="season")
    .filter((pl.col("n_rows") < pl.col("median_rows") * 0.5))
    .sort(["season", "n_rows"])
)

print(f"\nTeam con copertura < 50% della mediana stagionale: {anomalous.shape[0]}")
if anomalous.shape[0] > 0:
    show(anomalous.to_pandas(), caption="Team con copertura anomala")

---
### 5. Variabili Categoriche

#### 5.1 Bilanciamento W / L

In [ ]:
wl_counts = (
    df.group_by("wl")
      .agg(pl.len().alias("n"))
      .sort("wl")
)

total = wl_counts["n"].sum()
wl_counts = wl_counts.with_columns(
    (pl.col("n") / total * 100).round(2).alias("pct")
)

print("Distribuzione W/L:")
for row in wl_counts.iter_rows(named=True):
    print(f"  {row['wl']}: {row['n']:,}  ({row['pct']:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors_wl = ["#1d428a", "#c8102e"]
axes[0].bar(wl_counts["wl"].to_list(), wl_counts["n"].to_list(),
            color=colors_wl, edgecolor="white")
axes[0].axhline(total / 2, color=NEUTRAL, linestyle="--", linewidth=1,
                label="50% line")
axes[0].set_title("Conteggio W/L")
axes[0].set_ylabel("N. righe")
axes[0].legend(fontsize=9)

# Pie chart
axes[1].pie(
    wl_counts["n"].to_list(),
    labels=[f"{r['wl']} ({r['pct']:.1f}%)" for r in wl_counts.iter_rows(named=True)],
    colors=colors_wl,
    startangle=90,
    wedgeprops={"edgecolor": "white"},
)
axes[1].set_title("Proporzione W/L")

fig.suptitle("Bilanciamento target W/L (ogni partita ha 1 W + 1 L → 50/50 atteso)",
             fontsize=12)
plt.tight_layout()
plt.show()

#### 5.2 Bilanciamento Home / Away

In [ ]:
ha_counts = (
    df.group_by("home_away")
      .agg(pl.len().alias("n"))
      .sort("home_away")
)

total_ha = ha_counts["n"].sum()
ha_counts = ha_counts.with_columns(
    (pl.col("n") / total_ha * 100).round(2).alias("pct")
)

print("Distribuzione Home/Away:")
for row in ha_counts.iter_rows(named=True):
    print(f"  {row['home_away']}: {row['n']:,}  ({row['pct']:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors_ha = ["#007a33", "#ce1141"]
axes[0].bar(ha_counts["home_away"].to_list(), ha_counts["n"].to_list(),
            color=colors_ha, edgecolor="white")
axes[0].axhline(total_ha / 2, color=NEUTRAL, linestyle="--", linewidth=1,
                label="50% line")
axes[0].set_title("Conteggio Home/Away")
axes[0].set_ylabel("N. righe")
axes[0].legend(fontsize=9)

axes[1].pie(
    ha_counts["n"].to_list(),
    labels=[f"{r['home_away']} ({r['pct']:.1f}%)" for r in ha_counts.iter_rows(named=True)],
    colors=colors_ha,
    startangle=90,
    wedgeprops={"edgecolor": "white"},
)
axes[1].set_title("Proporzione Home/Away")

fig.suptitle("Bilanciamento Home/Away (atteso esattamente 50/50)", fontsize=12)
plt.tight_layout()
plt.show()

#### 5.3 Distribuzione righe per SEASON_YEAR

In [ ]:
season_counts = (
    df.group_by("season")
      .agg(pl.len().alias("n_rows"))
      .sort("season")
)

fig, ax = plt.subplots(figsize=(FIG_W_GRID, FIG_H_SINGLE))
ax.bar(season_counts["season"].to_list(),
       season_counts["n_rows"].to_list(),
       color=ACCENT, edgecolor="white", linewidth=0.4)
ax.axhline(season_counts["n_rows"].mean(), color="#fdb927",
           linestyle="--", linewidth=1.5,
           label=f"media={season_counts['n_rows'].mean():.0f}")
ax.set_xlabel("Stagione")
ax.set_ylabel("N. righe (team-game)")
ax.set_title("Righe per stagione (ogni partita = 2 righe)")
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

show(season_counts.to_pandas(), caption="Righe per stagione")

---
### 6. Analisi del Target — Win Rate Home Team

> **Obiettivo:** calcolare la percentuale storica di vittorie del team che gioca in casa.  
> Questo valore è la **baseline naive** che qualsiasi modello deve superare per essere utile.  
> La letteratura NBA indica circa **60%** di vittorie home nel lungo periodo.

In [ ]:
home_games = df.filter(pl.col("home_away") == "Home")

total_home  = home_games.shape[0]
home_wins   = home_games.filter(pl.col("wl") == "W").shape[0]
home_win_pct = home_wins / total_home * 100

print("=" * 55)
print(f"  Partite home totali    : {total_home:>10,}")
print(f"  Vittorie home          : {home_wins:>10,}")
print(f"  HOME WIN RATE (baseline): {home_win_pct:>10.2f}%")
print("=" * 55)
print(f"  Riferimento letteratura: ~60.0%")
print(f"  Delta vs letteratura   : {home_win_pct - 60.0:>+10.2f} pp")

#### 6.1 Home win rate per stagione

In [ ]:
home_wr_season = (
    home_games
    .group_by("season")
    .agg(
        pl.len().alias("n_home"),
        (pl.col("wl") == "W").sum().alias("n_wins"),
    )
    .with_columns(
        (pl.col("n_wins") / pl.col("n_home") * 100).round(2).alias("home_wr_pct")
    )
    .sort("season")
)

seasons_wr = home_wr_season["season"].to_list()
wr_vals    = home_wr_season["home_wr_pct"].to_list()

fig, ax = plt.subplots(figsize=(FIG_W_GRID, FIG_H_SINGLE))

bar_col = [
    "#1d428a" if v >= 50 else "#c8102e" for v in wr_vals
]
ax.bar(seasons_wr, wr_vals, color=bar_col, edgecolor="white", linewidth=0.4)
ax.axhline(home_win_pct, color="#fdb927", linewidth=2, linestyle="--",
           label=f"Overall baseline={home_win_pct:.1f}%")
ax.axhline(50.0, color=NEUTRAL, linewidth=1.2, linestyle=":")
ax.set_ylim(40, 70)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Stagione")
ax.set_ylabel("Home win rate (%)")
ax.set_title("Home win rate per stagione — baseline modello")
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nStd dev home win rate across seasons: {np.std(wr_vals):.2f} pp")
print(f"Min: {min(wr_vals):.1f}% ({seasons_wr[wr_vals.index(min(wr_vals))]})")
print(f"Max: {max(wr_vals):.1f}% ({seasons_wr[wr_vals.index(max(wr_vals))]})")

show(
    home_wr_season.to_pandas(),
    caption="Home win rate per stagione",
)

---
### 7. Ispezione Outlier

> Metodi usati: **IQR fence** (1.5× e 3×) e **z-score** (soglia |z| > 3.0).  
> I risultati vengono confrontati con il flag `is_outlier_flag` prodotto in `03_data_cleaning.ipynb`.

In [ ]:
outlier_summary = []

for col in ALL_NUM_VARS:
    series = df[col].drop_nulls().to_numpy().astype(float)
    q25, q75 = np.percentile(series, [25, 75])
    iqr = q75 - q25

    fence_mild  = (q25 - 1.5 * iqr, q75 + 1.5 * iqr)
    fence_ext   = (q25 - 3.0 * iqr, q75 + 3.0 * iqr)

    n_iqr_mild  = int(np.sum((series < fence_mild[0]) | (series > fence_mild[1])))
    n_iqr_ext   = int(np.sum((series < fence_ext[0])  | (series > fence_ext[1])))

    z_scores    = np.abs(stats.zscore(series))
    n_zscore    = int(np.sum(z_scores > Z_THRESHOLD))

    outlier_summary.append({
        "variable"      : col,
        "iqr"           : round(iqr, 4),
        "fence_low_1.5x": round(fence_mild[0], 4),
        "fence_high_1.5x": round(fence_mild[1], 4),
        "n_iqr_mild"    : n_iqr_mild,
        "n_iqr_extreme" : n_iqr_ext,
        "n_zscore_3s"   : n_zscore,
        "pct_iqr_mild"  : round(n_iqr_mild / len(series) * 100, 3),
        "pct_zscore_3s" : round(n_zscore   / len(series) * 100, 3),
    })

outlier_df = pl.DataFrame(outlier_summary)

show(
    outlier_df.to_pandas(),
    caption="Riepilogo outlier per variabile (IQR + z-score)",
)

#### 7.1 Confronto con is_outlier_flag

In [ ]:
n_flagged = df.filter(pl.col("is_outlier_flag") == True).shape[0]
n_total   = df.shape[0]

print(f"Righe con is_outlier_flag=True : {n_flagged:,}  ({n_flagged / n_total * 100:.2f}%)")
print(f"Righe con is_outlier_flag=False: {n_total - n_flagged:,}  ({(n_total - n_flagged) / n_total * 100:.2f}%)")

# Z-score union across all numeric vars
zscore_mask = np.zeros(df.shape[0], dtype=bool)
for col in ALL_NUM_VARS:
    null_mask = df[col].is_null().to_numpy()
    arr       = df[col].fill_null(df[col].mean()).to_numpy().astype(float)
    z         = np.abs(stats.zscore(arr))
    zscore_mask |= (z > Z_THRESHOLD) & ~null_mask

n_zscore_any = int(np.sum(zscore_mask))
print(f"\nRighe con almeno una variabile |z|>3 : {n_zscore_any:,}  ({n_zscore_any / n_total * 100:.2f}%)")

# Overlap
flag_arr = df["is_outlier_flag"].to_numpy()
both     = int(np.sum(flag_arr & zscore_mask))
only_flag= int(np.sum(flag_arr & ~zscore_mask))
only_z   = int(np.sum(~flag_arr & zscore_mask))

print(f"\nOverlap flag ∩ z-score   : {both:,}")
print(f"Solo flag (non z-score)  : {only_flag:,}")
print(f"Solo z-score (non flag)  : {only_z:,}")

#### 7.2 Visualizzazione outlier per variabili di performance

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(FIG_W_GRID, FIG_H_ROW * n_rows))
axes = axes.flatten()

for i, col in enumerate(PERF_VARS):
    ax      = axes[i]
    arr     = df[col].drop_nulls().to_numpy().astype(float)
    z       = stats.zscore(arr)
    inliers = arr[np.abs(z) <= Z_THRESHOLD]
    outliers= arr[np.abs(z) >  Z_THRESHOLD]

    ax.scatter(range(len(inliers)),  inliers,  s=1, alpha=0.3,
               color=ACCENT,    label=f"normal ({len(inliers):,})")
    ax.scatter(range(len(inliers), len(inliers) + len(outliers)),
               outliers, s=6, alpha=0.8,
               color="#c8102e", label=f"|z|>3 ({len(outliers):,})")

    ax.set_title(col.upper())
    ax.set_ylabel(col.upper())
    ax.set_xlabel("index")
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"Outlier |z|>{Z_THRESHOLD} per variabili di performance",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

#### 7.3 Record outlier — lista e decisione documentata

In [ ]:
import pandas as pd

# Collect rows that are extreme outliers on any of the PERF_VARS (|z|>3)
df_pandas = df.to_pandas()

extreme_mask = pd.Series([False] * len(df_pandas))
for col in PERF_VARS:
    col_z = np.abs(stats.zscore(df_pandas[col].dropna()))
    z_flags = pd.Series([False] * len(df_pandas), index=df_pandas.index)
    z_flags.loc[df_pandas[col].dropna().index] = col_z > Z_THRESHOLD
    extreme_mask = extreme_mask | z_flags

extreme_rows = df_pandas[extreme_mask][
    ["game_id", "game_date", "season", "team_abbreviation",
     "home_away", "wl"] + PERF_VARS + ["is_outlier_flag"]
].copy()
extreme_rows = extreme_rows.sort_values("pts", ascending=False)

print(f"Record outlier estremi (|z|>3 su almeno una variabile performance): {len(extreme_rows):,}")
show(extreme_rows.head(50), caption="Top 50 record outlier — performance variables")

#### 7.4 Decisione documentata

| Categoria | Decisione | Motivazione |
|-----------|-----------|-------------|
| **PTS < 70** | Mantenere + flag | Partite storicamente reali (bubble 2020, overtime, squalifiche) — nessuna evidenza di errore |
| **PTS > 160** | Mantenere + flag | Partite ad alto punteggio avvenute realmente (Nuggets-Suns 2023, ecc.) |
| **OFF/DEF RATING outlier** | Mantenere + flag | Valori estremi correlati a stagioni abbreviate (bubble, lockout) — eventi reali |
| **PACE outlier** | Mantenere + flag | Correlato a stile di gioco dell'epoca (2004-05: pace slowdown) — evento reale |
| **TS_PCT outlier** | Verificare | Valori > 0.90 o < 0.30 richiedono verifica manuale — possibile errore di encoding |

> **Conclusione generale:** nessun record viene rimosso in questa fase.  
> Gli outlier sono eventi reali estremi, non errori di acquisizione dati.  
> Il flag `is_outlier_flag` viene propagato al Feature Engineering per decisioni model-specific.

In [ ]:
# Quick sanity check on TS_PCT extreme values
ts_extreme = df.filter(
    (pl.col("ts_pct") > 0.90) | (pl.col("ts_pct") < 0.30)
).select(
    ["game_date", "season", "team_abbreviation", "home_away", "wl",
     "pts", "fga", "fg3a", "fta", "ts_pct", "is_outlier_flag"]
).sort("ts_pct")

print(f"Righe con TS_PCT < 0.30 o > 0.90: {ts_extreme.shape[0]}")
if ts_extreme.shape[0] > 0:
    show(ts_extreme.to_pandas(), caption="TS_PCT valori estremi — verifica manuale")

---
### Riepilogo EDA Univariata

| Area | Trovato | Azione |
|------|---------|--------|
| **Skewness** | Variabili con \|skew\| > 1 (da output sezione 2) | Candidati trasformazione log nel Feature Engineering |
| **Kurtosis** | Variabili con kurtosis excess > 3 | Code pesanti → modelli robusti agli outlier |
| **Copertura stagionale** | Stagioni abbreviate (COVID 2019-20, lockout 2011-12) | Flag già presenti — da tenere in considerazione |
| **Bilanciamento W/L** | Esattamente 50/50 (per costruzione dataset) | Nessun problema di class imbalance a livello row |
| **Home/Away** | Esattamente 50/50 | Corretto — ogni partita ha 1 home + 1 away |
| **Baseline target** | Home win rate ≈ 60% | Baseline naive del modello classificatore |
| **Outlier** | Tutti eventi reali estremi | Mantenere + propagare `is_outlier_flag` |